# Static and dynamic topic modeling with BERTopic (1987–2002)

Fits a BERTopic model on the documents of one time period and analyses how its topics develop over time.

1. Embed every document with `Alibaba-NLP/gte-base-en-v1.5`.
2. Fit a static BERTopic model (UMAP, HDBSCAN, c-TF-IDF) with KeyBERT-inspired and MMR topic representations.
3. Reduce outliers.
4. Compute and visualise topics over time, using the publication year of each document.

The input is the fully preprocessed text of the period, produced by the notebooks in `notebooks/Data_preparation_subsets/`.

In [ ]:
import pandas as pd
import joblib
import collections
import numpy
import matplotlib.pyplot as plt
import time
import swifter
import spacy

from nltk.tokenize import sent_tokenize,word_tokenize
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
from wordcloud import WordCloud

# # Load spaCy model
nlp = spacy.load("en_core_web_sm")

In [ ]:
!python --version

In [ ]:
import bertopic
bertopic.__version__

In [ ]:
data_1987_2002_df= pd.read_pickle("../../datasets/data_split_timeframe/data_1987_2002_cleaned_df_regex_lemma_stop.pkl") # word_embedding_data(preprocesed)

In [ ]:
def preprocess(text):
    doc = nlp(text.lower())
    tokens = [token.text for token in doc if token.is_alpha]  # Remove punctuation and special characters
    return tokens
data_1987_2002_df['tokens'] = data_1987_2002_df['content_cleaned_spacy_lemmatized_cleaned'].swifter.apply(preprocess)

In [ ]:
data_1987_2002_df['content_cleaned_spacy_lemmatized_cleaned'] = data_1987_2002_df['content_cleaned_spacy_lemmatized_cleaned'].str.lower()

In [ ]:
data_1987_2002_df["num_tokens_cleaned"] = data_1987_2002_df["content_cleaned_spacy_lemmatized_cleaned"].swifter.apply(lambda content: len(word_tokenize(content)))

In [ ]:
data_1987_2002_df['num_tokens_cleaned'].hist(bins=60)
data_1987_2002_df['num_tokens_cleaned'].describe()

In [ ]:
# model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True)
# model.max_seq_length

In [ ]:
data_1987_2002_df.to_pickle("../../datasets/data_split_timeframe/data_1987_2002_cleaned_df_regex_lemma_stop.pkl")

## Compute the document embeddings

In [ ]:
# Requires sentence_transformers>=2.7.0
model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True)

data_1987_2002_df["sentence_embeddings"] = data_1987_2002_df["content_cleaned_spacy_lemmatized_cleaned"].swifter.apply(lambda doc: model.encode(doc))

In [ ]:
data_1987_2002_df.to_pickle("../../datasets/gte_bertopic_embeddings/data_1987_2002_embeddings_df.pkl")

In [ ]:
data_1987_2002_df= pd.read_pickle("../../datasets/gte_bertopic_embeddings/data_1987_2002_embeddings_df.pkl")

In [ ]:
 # Hyperparameters: {'min_cluster_size': [31], 'min_samples': [24], 'min_topic_size': [37], 'n_components': [5], 'n_neighbors': [10]}, Objective Value: -0.7670762290091766

## Static topic model

BERTopic is fitted once on all documents of the period, using the precomputed embeddings.

In [ ]:
embeddings = data_1987_2002_df["sentence_embeddings"].to_list()
contents = data_1987_2002_df["content_cleaned_spacy_lemmatized_cleaned"].to_list()
years = data_1987_2002_df["year"].to_list()
print(len(embeddings), len(contents) , len(years))

In [ ]:
contents, embeddings , years = numpy.array(contents), numpy.array(embeddings) , numpy.array(years)
print(contents.shape, embeddings.shape , years.shape)

In [ ]:
embedding_model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True)

In [ ]:
# Extract vocab to be used in BERTopic
vocab_counter = collections.Counter()
tokenizer = CountVectorizer().build_tokenizer()
# for doc in tqdm(all_embeddings_df["file_content"]):
#   vocab_counter.update(tokenizer(doc))
# vocab = [word for word, frequency in vocab_counter.items() if frequency >= 100]; len(vocab)

In [ ]:
# Train model and reduce dimensionality of gte_bertopic_embeddings
umap_model = UMAP(n_components=5, n_neighbors=10, random_state=42, metric="cosine", verbose=True)
# reduced_embeddings = umap_model.fit_transform(embeddings)
# Train model and reduce dimensionality of gte_bertopic_embeddings
# umap_2d_model = UMAP(n_components=2, n_neighbors=5, random_state=42, metric="cosine", verbose=True)
# # reduced_embeddings_2d = umap_model.fit_transform(gte_bertopic_embeddings)

In [ ]:
# If you want many highly specific clusters, use a small min_samples and a small min_cluster_size.
# If you want more generalized clusters but still want to keep most detail, use a small min_samples and a large min_cluster_size
# If you want very very general clusters and to discard a lot of noise in the clusters, use a large min_samples and a large min_cluster_size.

In [ ]:
# Find clusters of semantically similar documents
#hdbscan_model = HDBSCAN(min_samples=10, gen_min_span_tree=True, prediction_data=True, min_cluster_size=20)
hdbscan_model = HDBSCAN( min_samples=24,gen_min_span_tree=True, prediction_data=True, min_cluster_size=31)
# clusters = hdbscan_model.fit(reduced_embeddings).labels_

In [ ]:
# vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(2, 3))
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 1), min_df=10)

In [ ]:
# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
# pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    # "POS": pos_model
}

In [ ]:
start_time = time.time()

print("Start",start_time)

topic_model = BERTopic(
  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,
  # Hyperparameters
  top_n_words=10,
  verbose=True, calculate_probabilities=True,low_memory=True
)
topics,probs=topic_model.fit_transform(contents, embeddings)

end_time = time.time()
print("End",end_time)
execution_time = end_time - start_time

print("Execution time:", execution_time, "seconds")

In [ ]:
topic_model.save("../../models/gte_bertopic_models/safetensors/1987_2002", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
topic_model.save("../../models/gte_bertopic_models/data_1987_2002_embeddings_bertopic_model.pkl", serialization="pickle")

In [ ]:
# loaded_model = BERTopic.load("../../models/gte_bertopic_models/safetensors/1987_2002",embedding_model=embedding_model)
# loaded_model.get_topic_info()

In [ ]:
# joblib.dump(topic_model, "../../models/gte_bertopic_models/data_1987_2002_embeddings_bertopic_model.pkl")
# topic_model.get_topic_info()

### Load the saved model

In [ ]:

bertopic_models= BERTopic.load("../../models/gte_bertopic_models/data_1987_2002_embeddings_bertopic_model.pkl")
bertopic_models.get_topic_info()

## Reduce outliers

In [ ]:
# reduce outliers (try)
import hdbscan

topics= bertopic_models._map_predictions(bertopic_models.hdbscan_model.labels_)

probs = hdbscan.all_points_membership_vectors(bertopic_models.hdbscan_model)
probs = bertopic_models._map_probabilities(probs, original_topics=True)

bertopic_modelss= bertopic_models

new_topics= bertopic_modelss.reduce_outliers(contents,topics,probabilities=probs, strategy="probabilities")

bertopic_modelss.update_topics(contents, topics=new_topics)
bertopic_modelss.get_topic_info()

In [ ]:
bertopic_modelss.save("../../models/gte_bertopic_models/data_1987_2002_embeddings_bertopic_model_outlier_reduced.pkl", serialization="pickle")

In [ ]:
bertopic_models.generate_topic_labels(10)

## Topic assignments and further visualisations

In [ ]:
document_info= bertopic_models.get_document_info(contents)

document_info.to_pickle("../../Document_info_with_topics/document_info_with_topics_1987_2002.pkl")
document_info

In [ ]:
bertopic_models.visualize_distribution(topic_model.probabilities_[3])

## Dynamic topic modeling: topics over time

The fitted model is not retrained. The documents are grouped by publication year, and the frequency and the representation of each topic are computed per year.

In [ ]:
topics_over_time = bertopic_models.topics_over_time(contents, years)

In [ ]:
joblib.dump(topics_over_time, "../../models/gte_bertopic_models/data_1987_2002_embeddings_Dynamic_bertopic_model.pkl")

In [ ]:
topics_over_timee = joblib.load("../../models/gte_bertopic_models/data_1987_2002_embeddings_Dynamic_bertopic_model.pkl")
topic_over_time_vis = bertopic_models.visualize_topics_over_time(topics_over_timee, top_n_topics=170,
                                                                 title='</b>Topic over time (1987_2002) </b>')
topic_over_time_vis.write_html("../../Figures/data_time_frames/visualize_topic_over_time_1987_2002.html")

In [ ]:
fig1= bertopic_models.visualize_barchart(top_n_topics=190,n_words=10)
fig1.write_html("../../Figures/data_time_frames/barchart/visualize_barchart_1987_2002.html")

In [ ]:
hierarchical_topics = bertopic_models.hierarchical_topics(contents)
hierarchical_topics

In [ ]:

fig2=bertopic_models.visualize_hierarchy(title="1987_2002",hierarchical_topics=hierarchical_topics)
fig2.write_html("../../Figures/data_time_frames/hierarchy/visualize_hierarchy_1987_2002.html")

In [ ]:
# # outlier reduction
# bertopic_models_reduced=bertopic_models
# topic_list = bertopic_models_reduced.topics_
# probabilities = bertopic_models_reduced.probabilities_
# # Reduce outliers using the `probabilities` strategy
# new_topics = bertopic_models_reduced.reduce_outliers(contents, topic_list, probabilities=probabilities,
#                                                      strategy="probabilities", threshold=0.05)
# new_topics
# bertopic_models_reduced.update_topics(contents, new_topics)
# bertopic_models_reduced.get_topic_info()